# Episode 3: Give Your Agent Tools

An agent that can only talk is like an employee who can only give advice but never touch a keyboard.

**In this episode, you'll build:** An agent that can check the weather using a tool.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from autogen import ConversableAgent, LLMConfig
from typing import Annotated

## Why tools?

LLMs are impressive reasoners, but they live inside a text box. They can't browse the web, query a database, or call an API on their own.

**Tools** are the bridge between reasoning and action. In AG2, tools are just Python functions with type hints — the framework handles all the plumbing: describing the tool to the LLM, parsing the request, calling your function, and returning the result. This is what separates real production systems from fancy chatbots. Gartner predicts 40% of enterprise apps will embed tool-using AI agents by the end of 2026.

## Step 1: Define a tool

A tool is just a Python function. Two things make it work with AG2:

- **Type annotations** — these become the tool's input schema
- **A docstring** — this tells the LLM what the tool does

Let's see this in action:

In [ ]:
from typing import Annotated

def get_weather(city: Annotated[str, "The city to check weather for"]) -> str:
    """Get the current weather for a city."""
    # Mock weather data - in production, call a real API
    weather_data = {
        "new york": "72°F, Sunny",
        "london": "59°F, Cloudy",
        "tokyo": "68°F, Clear",
    }
    city_lower = city.lower()
    if city_lower in weather_data:
        return f"Weather in {city}: {weather_data[city_lower]}"
    return f"Weather in {city}: 65°F, Partly Cloudy (default)"

## Step 2: Create agents and register the tool

Registering a tool is a two-step process, and the two steps go on different agents:

- **`register_for_llm`** on the **weather agent** — tells the LLM the tool exists, so it can decide when to call it
- **`register_for_execution`** on the **user agent** — the user agent actually runs the function when the LLM requests a tool call

Why split it? Because the agent that *thinks* ("I should check the weather") is different from the agent that *acts* (actually calling the function). This separation becomes important in multi-agent systems where one agent suggests an action and another executes it.

In [ ]:
from autogen import ConversableAgent, LLMConfig

# Connect to OpenAI's gpt-4o-mini model
llm_config = LLMConfig({"model": "gpt-4o-mini"})

weather_agent = ConversableAgent(
    name="weather_agent",
    system_message="You are a weather assistant. Use the get_weather tool to answer weather questions.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

user = ConversableAgent(name="user", human_input_mode="NEVER")

# Tell the LLM about the tool (so it knows it can call it)
weather_agent.register_for_llm(description="Get weather for a city")(get_weather)

# Register execution on the user agent (it runs the tool when the LLM suggests a call)
user.register_for_execution()(get_weather)

## Step 3: Ask about the weather

Same pattern as Episode 2 — we create a `user` agent to start the conversation, then let the weather agent respond. Watch what happens: the agent will decide on its own to call the `get_weather` tool.

In [ ]:
result = user.run(weather_agent, message="What's the weather like in Tokyo?", max_turns=3)
result.process()
print(result.summary)

## How tools work

Notice that the agent *decided* to call the tool — you didn't tell it to. You asked a question in plain English, and the agent recognized that `get_weather` was the right tool for the job.

Here's the full flow of what just happened:

1. **You sent a message** — "What's the weather in Tokyo?"
2. **The LLM decided to call `get_weather`** — it recognized the tool was relevant
3. **The agent executed the function** — `get_weather("Tokyo")` ran locally on your machine
4. **The result went back to the LLM** — it saw `"Weather in Tokyo: 68°F, Clear"`
5. **The LLM formulated a response** — wrapping the raw data in a natural language answer

This tool call flow is the foundation of how agents take real-world actions.

## ReplyResult: How tools return data

For now, simple return values like strings work perfectly. But as your agents get more sophisticated, you'll want tools that can return structured data *and* control which agent speaks next.

That's what **`ReplyResult`** is for — this becomes important starting in Episode 7, so keep it in the back of your mind.

## Try it yourself

What happens if you give the agent a *second* tool? Try adding a `get_time(city)` function that returns the current time for a city. Register it, then ask:

> "What's the weather and time in London?"

```python
def get_time(city: Annotated[str, "The city to check the time for"]) -> str:
    """Get the current time for a city."""
    # Your implementation here
    ...
```

Does the agent figure out it needs to call both tools?

## Additional Content

### The LLM/executor split

Earlier I mentioned there's a reason for the two registration steps. In multi-agent systems, **one agent might suggest a tool call while a different agent executes it**.

Picture a "planner" agent that decides which tool to call (registered for LLM) and a "worker" agent that actually runs the function (registered for execution). This separation enables powerful delegation patterns that we'll explore in later episodes.

## What's Next

You've got an agent that can think *and* act. But one agent still means one perspective. What if you had two agents that could check each other's work?

In the next episode, you'll add a second agent — one researches, one reviews — and watch them collaborate.

**Bonus:** Try the weather demo in the [AG2 Playground](https://playground.ag2.ai).